# Leakage-free feature selection (inside every CV fold)

**Problem with the old notebook:** every selection step (univariate tests, mutual information, lasso, RF importance, VIF, ...) was fitted on the *whole* training set, and only afterwards were feature sets compared with cross-validation. The held-out fold had therefore already influenced which features were picked, so the CV scores were optimistic.

**What this notebook does instead**

1. Reuses the code of `preprocessing_pipeline.ipynb` (`CatBinner`, `RoundImputed`, `build_pipeline`, `build_simple_pipeline`) as the preprocessing front end.
2. Builds one `Pipeline` = *preprocessing → `FoldFeatureSelector` → final model* (`build_cv_pipeline`).
3. Runs `cross_validate` on that pipeline. In every fold the bin-rare-levels / imputation / encoding / scaling **and** the feature selection are fitted on the training part of the fold only; the validation part is only ever transformed.
4. Compares selection methods on the honest outer-fold scores and measures how *stable* each method's choices are across folds.
5. Only at the very end fits the selection once on the full training set to get the final feature list. The test set is never touched.

**Selection methods (all computed per fold, at the level of the *original* variable, i.e. dummies of one categorical are kept/dropped together):**

| method | idea |
|---|---|
| `none` | baseline, keep everything |
| `lasso` | L1 logistic regression, C picked by inner CV (best AUC); keep variables with a non-zero coefficient |
| `lasso_1se` | same, but the sparsest C within 1 SE of the best AUC (usually much sparser) |
| `rf_impurity` | random forest impurity importance vs. a random-noise column |
| `rf_perm` | random forest *grouped* permutation importance on an inner hold-out split |
| `mutual_info` | mutual information vs. the max MI obtained with a shuffled target |
| `univariate_lrt` | per-variable likelihood-ratio test vs. intercept-only, Benjamini–Hochberg adjusted |
| `glmm_wald` | random-intercept logistic regression (District = random effect): joint Wald test per variable given the District effect, Benjamini–Hochberg adjusted. District itself is always kept |
| `glmm_perm` | same random-effects model, grouped permutation AUC drop on an inner hold-out; predictions include each stop's District effect |
| `consensus` | keep variables chosen by at least `CONSENSUS_MIN_VOTES` of: univariate_lrt, mutual_info, lasso, rf_impurity, rf_perm |
| `consensus_glmm` | majority vote (4 of 7): the five consensus voters plus `glmm_wald` and `glmm_perm` |

In [ ]:
import json
import shutil
import tempfile
import warnings
from scipy import sparse
from scipy.stats import chi2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Memory

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_rows", 200, "display.float_format", "{:.4f}".format)

# ----------------------------- CONFIG ------------------------------------- #
TRAIN_PATH = r"/data/cleaned_data/stop_search_train1.csv"   # same path as preprocessing_pipeline.ipynb
SEED = 10
N_SPLITS = 5

# "simple" = build_simple_pipeline (SimpleImputer, fast). "full" = build_pipeline (IterativeImputer, slow;
# preprocessing is cached across selectors via joblib so it is only fitted once per fold).
PIPELINE_KIND = "simple"

# Columns dropped before modelling. Copied from preprocessing_pipeline.ipynb.
# NOTE: "year", "Zone", "Latitude", "Longitude" were strong candidates in the old selection notebook;
# remove them from this list if you want them to compete here too.
DROP_COLS = ["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI",
             "year", "Zone", "Latitude", "Longitude", "SubjectID"]

CONSENSUS_MIN_VOTES = 3      # default for "consensus" (3 of its 5 voters)

# name -> FoldFeatureSelector arguments. Add / remove entries freely.
SELECTOR_CONFIGS = {
    "none":           dict(method="none"),
    "lasso":          dict(method="lasso"),
    "lasso_1se":      dict(method="lasso_1se"),
    "rf_impurity":    dict(method="rf_impurity"),
    "rf_perm":        dict(method="rf_perm"),
    "mutual_info":    dict(method="mutual_info"),
    "univariate_lrt": dict(method="univariate_lrt"),
    # random-effects logistic regression, District as the random intercept
    "glmm_wald":      dict(method="glmm_wald", re_group="District"),
    "glmm_perm":      dict(method="glmm_perm", re_group="District"),
    "consensus":      dict(method="consensus"),
    "consensus_glmm": dict(method="consensus", re_group="District", min_votes=4,
                           voting_methods=("univariate_lrt", "mutual_info", "lasso", "rf_impurity",
                                           "rf_perm", "glmm_wald", "glmm_perm")),
}
SELECTION_METHODS = list(SELECTOR_CONFIGS)
STABLE_FREQ = 0.8            # a variable is "stable" if selected in >= 80% of outer folds
FINAL_METHOD = "consensus"   # method used for the final full-train selection

## 1. Data + column definitions (from `preprocessing_pipeline.ipynb`)
Only the **training** file is loaded. The test set stays untouched.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)     # NOT index_col=0: in this file the first column is District
# drop a genuine unnamed index column if the file has one
train_df = train_df.drop(columns=[c for c in train_df.columns if str(c).startswith("Unnamed")], errors="ignore")

train_df = train_df.drop(columns=DROP_COLS, errors="ignore")
train_df["Zip"] = train_df["Zip"].astype("Int64")

cat_cols = ["VehicleStyle", "VehicleColor", "SubjectRace", "SubjectEyeColor", "SubjectHairColor",
            "Zip", "StopDescription", "District"]
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "isLA", "VehicleYear",
            "SubjectHasPhotoID", "hour", "night", "month", "weekend", "isDriver"]
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "month", "hour"]
bin_cols = ["SubjectGender", "isLA", "SubjectHasPhotoID", "weekend", "night", "isDriver"]

# keep only columns that still exist (in case DROP_COLS / the data changed)
cat_cols = [c for c in cat_cols if c in train_df]
dis_cols = [c for c in dis_cols if c in train_df]
scale_cols = [c for c in scale_cols if c in train_df]
bin_cols = [c for c in bin_cols if c in train_df]

train_df[cat_cols] = train_df[cat_cols].astype(str)

target = "search_occurred"
X_train = train_df.drop(columns=[target])
y_train = train_df[target].astype(int)

listed = set(cat_cols) | set(scale_cols) | set(bin_cols)
remainder_cols = [c for c in X_train.columns if c not in listed]

# Columns that are in none of the lists are passed through the pipeline untouched. Free-text / identifier
# columns (e.g. an address string) cannot be used by the models, so they are dropped here.
text_remainder = [c for c in remainder_cols if not pd.api.types.is_numeric_dtype(X_train[c])]
if text_remainder:
    print("Dropping non-numeric columns not covered by any column list:", text_remainder)
    X_train = X_train.drop(columns=text_remainder)
    remainder_cols = [c for c in remainder_cols if c not in text_remainder]

print(f"Rows: {len(X_train)} | positive rate: {y_train.mean():.3%} | events: {y_train.sum()}")
print("Numeric columns passed through untouched (remainder):", remainder_cols or "none")

## 2. Preprocessing classes + pipeline builders (from `preprocessing_pipeline.ipynb`)
Unchanged except: the unused `narwhals` import was removed.

In [ ]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()
        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self


class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self


def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    ordinal = ColumnTransformer([
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat_cols)],
        remainder="passthrough", verbose_feature_names_out=False)

    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10),
                               skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    encode = ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                                ("scale", StandardScaler(), scale_cols)],
                               remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("ordinal", ordinal),
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    pipeline[:-2].set_output(transform="pandas")
    return pipeline


def build_simple_pipeline(model, cat_cols, bin_cols, scale_cols):
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    bin_pipe = SimpleImputer(strategy="most_frequent", add_indicator=True)

    preprocess = ColumnTransformer([
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, scale_cols),
        ("bin", bin_pipe, bin_cols),
    ], remainder="passthrough")

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("preprocess", preprocess),
        ("model", model)
    ])
    return pipeline

## 3. In-fold feature selector

Two small transformers are added on top of the preprocessing pipeline:

* `NamedTransform` wraps the last encoding step so it returns a **DataFrame with feature names** (dense). The selector needs the names to map one-hot dummies / missing-indicators back to their original variable. Any NaN left in pass-through columns is filled with training-fold medians.
* `FoldFeatureSelector` is a normal scikit-learn transformer: `fit` runs the chosen selection method on the data it is given (= the training part of a fold) and `transform` just drops columns. Selection is at the level of the original variable.

In [ ]:
def feature_group(name, cat_cols):
    """Map an encoded column name back to its original variable."""
    n = name.split("__", 1)[-1]                      # strip ColumnTransformer prefix (cat__, num__, ...)
    if n.startswith("missingindicator_"):
        n = n[len("missingindicator_"):]
    for c in sorted(cat_cols, key=len, reverse=True):
        if n == c or n.startswith(c + "_"):
            return c
    return n


def benjamini_hochberg(p):
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order] * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out


class NamedTransform(BaseEstimator, TransformerMixin):
    """Wrap a ColumnTransformer: dense DataFrame output with feature names, no leftover NaN."""
    def __init__(self, transformer):
        self.transformer = transformer

    def _to_frame(self, out, index):
        if sparse.issparse(out):
            out = out.toarray()
        return pd.DataFrame(np.asarray(out, dtype=float), index=index, columns=self.names_)

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def fit_transform(self, X, y=None, **fit_params):
        self.transformer_ = clone(self.transformer)
        out = self.transformer_.fit_transform(X, y)
        self.names_ = np.asarray(self.transformer_.get_feature_names_out(), dtype=object)
        df = self._to_frame(out, X.index)
        self.fill_ = df.median().fillna(0.0)          # training-fold medians only
        return df.fillna(self.fill_)

    def transform(self, X):
        df = self._to_frame(self.transformer_.transform(X), X.index)
        return df.fillna(self.fill_)

    def get_feature_names_out(self, input_features=None):
        return self.names_


class FoldFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Feature selection at the level of the original variable. Everything is learned in `fit`
    from the rows it receives, so inside a CV Pipeline it only ever sees the training fold.

    method: none | lasso | lasso_1se | rf_impurity | rf_perm | mutual_info | univariate_lrt
            | glmm_wald | glmm_perm | consensus
    glmm_*: logistic regression with a random intercept for `re_group` (District). The grouping
            variable is not a fixed effect; with re_keep_group=True it is always kept as a feature.
    After fit:
      selected_groups_ : list of kept original variables
      scores_          : {method: Series(group -> importance, higher = more important)}
      selected_by_     : {method: list of kept groups}   (for consensus: includes every voter)
    """
    def __init__(self, method="lasso", cat_cols=(),
                 voting_methods=("univariate_lrt", "mutual_info", "lasso", "rf_impurity", "rf_perm"),
                 min_votes=3, inner_cv=3, lasso_Cs=None,
                 n_estimators=150, rf_min_leaf=20, rf_max_depth=14,
                 perm_repeats=5, perm_rows=8000, mi_rows=30000,
                 alpha=0.05, n_jobs=-1, random_state=10,
                 re_group="District", re_keep_group=True, glmm_prior_sd=2.0):
        self.method = method
        self.cat_cols = cat_cols
        self.voting_methods = voting_methods
        self.min_votes = min_votes
        self.inner_cv = inner_cv
        self.lasso_Cs = lasso_Cs
        self.n_estimators = n_estimators
        self.rf_min_leaf = rf_min_leaf
        self.rf_max_depth = rf_max_depth
        self.perm_repeats = perm_repeats
        self.perm_rows = perm_rows
        self.mi_rows = mi_rows
        self.alpha = alpha
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.re_group = re_group
        self.re_keep_group = re_keep_group
        self.glmm_prior_sd = glmm_prior_sd

    # ------------------------------------------------------------------ fit
    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("FoldFeatureSelector needs a DataFrame (use NamedTransform before it).")
        yv = np.asarray(y).astype(int)
        Xv = X.to_numpy(dtype=float)
        self.columns_ = list(X.columns)
        self.group_of_ = np.array([feature_group(c, self.cat_cols) for c in self.columns_], dtype=object)
        self.groups_ = list(dict.fromkeys(self.group_of_))
        self._gidx = {g: np.where(self.group_of_ == g)[0] for g in self.groups_}
        rng = np.random.default_rng(self.random_state)

        self.scores_, self.selected_by_ = {}, {}
        cache = {}
        if self.method == "consensus":
            to_run = list(self.voting_methods)
        elif self.method == "none":
            to_run = []
        else:
            to_run = [self.method]
        for m in to_run:
            self._run(m, Xv, yv, rng, cache)

        if self.method == "none":
            selected = list(self.groups_)
        elif self.method == "consensus":
            votes = pd.Series(0, index=self.groups_, dtype=float)
            for m in self.voting_methods:
                votes[self.selected_by_[m]] += 1
            self.scores_["consensus"] = votes
            selected = votes[votes >= self.min_votes].index.tolist()
            self.selected_by_["consensus"] = selected
        else:
            selected = self.selected_by_[self.method]

        self.fallback_ = False
        if len(selected) == 0:
            warnings.warn(f"{self.method}: nothing selected in this fold, keeping all variables.")
            selected, self.fallback_ = list(self.groups_), True

        keep = set(selected)
        self.selected_groups_ = [g for g in self.groups_ if g in keep]
        self.support_ = np.array([g in keep for g in self.group_of_])
        return self

    def transform(self, X):
        return X.loc[:, self.support_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columns_, dtype=object)[self.support_]

    # ------------------------------------------------------------ dispatch
    def _run(self, m, Xv, yv, rng, cache):
        if m == "lasso":
            self._lasso(Xv, yv, cache, rule="best")
        elif m == "lasso_1se":
            self._lasso(Xv, yv, cache, rule="1se")
        elif m in ("rf_impurity", "rf_perm"):
            self._rf(m, Xv, yv, rng, cache)
        elif m == "mutual_info":
            self._mutual_info(Xv, yv, rng)
        elif m == "univariate_lrt":
            self._univariate(Xv, yv)
        elif m in ("glmm_wald", "glmm_perm"):
            self._glmm(m, Xv, yv, rng, cache)
        else:
            raise ValueError(f"Unknown method: {m}")

    def _by_group(self, per_column):
        s = pd.Series(np.asarray(per_column, dtype=float)).groupby(self.group_of_)
        return s

    # --------------------------------------------------------------- lasso
    def _lasso(self, Xv, yv, cache, rule):
        if "lasso" not in cache:
            Xs = StandardScaler().fit_transform(Xv)            # scaling learned on the training fold only
            Cs = np.logspace(-3, 0, 8) if self.lasso_Cs is None else np.asarray(self.lasso_Cs)
            inner = StratifiedKFold(self.inner_cv, shuffle=True, random_state=self.random_state)
            cv_model = LogisticRegressionCV(Cs=Cs, cv=inner, penalty="l1", solver="liblinear",
                                            scoring="roc_auc", tol=1e-3, max_iter=500,
                                            random_state=self.random_state, n_jobs=1).fit(Xs, yv)
            cache["lasso"] = (cv_model, Xs, Cs)
        cv_model, Xs, Cs = cache["lasso"]

        if rule == "best":
            coef, name = cv_model.coef_[0], "lasso"
            self.lasso_C_ = float(cv_model.C_[0])
        else:
            sc = cv_model.scores_[1]                              # (n_inner_folds, n_Cs)
            mean = sc.mean(axis=0)
            se = sc.std(axis=0, ddof=1) / np.sqrt(sc.shape[0])
            best = int(mean.argmax())
            ok = np.where(mean >= mean[best] - se[best])[0]
            idx = int(ok.min())                                   # smallest C = strongest penalty
            refit = LogisticRegression(penalty="l1", solver="liblinear", C=float(Cs[idx]),
                                       tol=1e-3, max_iter=500).fit(Xs, yv)
            coef, name = refit.coef_[0], "lasso_1se"
            self.lasso_1se_C_ = float(Cs[idx])

        score = self._by_group(np.abs(coef)).sum().reindex(self.groups_)
        self.scores_[name] = score
        self.selected_by_[name] = score[score > 0].index.tolist()

    # ------------------------------------------------------ random forest
    def _rf(self, m, Xv, yv, rng, cache):
        if "rf" not in cache:
            Xtr, Xva, ytr, yva = train_test_split(Xv, yv, test_size=0.25, stratify=yv,
                                                  random_state=self.random_state)
            # one random-noise column is appended as a yardstick
            Xtr = np.column_stack([Xtr, rng.random(len(Xtr))])
            Xva = np.column_stack([Xva, rng.random(len(Xva))])
            rf = RandomForestClassifier(n_estimators=self.n_estimators, min_samples_leaf=self.rf_min_leaf,
                                        max_depth=self.rf_max_depth, max_samples=0.5,
                                        n_jobs=self.n_jobs, random_state=self.random_state).fit(Xtr, ytr)
            cache["rf"] = (rf, Xva, yva)
        rf, Xva, yva = cache["rf"]

        if m == "rf_impurity":
            imp = rf.feature_importances_
            noise = imp[-1]
            score = self._by_group(imp[:-1]).sum().reindex(self.groups_)     # summed over dummies
            self.scores_["rf_impurity"] = score
            self.selected_by_["rf_impurity"] = score[score > noise].index.tolist()
            self.rf_noise_impurity_ = float(noise)
            return

        # grouped permutation importance: all columns of one variable are shuffled together
        if len(yva) > self.perm_rows:
            sub = rng.choice(len(yva), self.perm_rows, replace=False)
            Xp, yp = Xva[sub], yva[sub]
        else:
            Xp, yp = Xva, yva
        base = roc_auc_score(yp, rf.predict_proba(Xp)[:, 1])
        mean_drop, std_drop = {}, {}
        for g, idx in self._gidx.items():
            drops = []
            for _ in range(self.perm_repeats):
                perm = rng.permutation(len(yp))
                Xq = Xp.copy()
                Xq[:, idx] = Xp[np.ix_(perm, idx)]
                drops.append(base - roc_auc_score(yp, rf.predict_proba(Xq)[:, 1]))
            mean_drop[g], std_drop[g] = float(np.mean(drops)), float(np.std(drops))
        score = pd.Series(mean_drop).reindex(self.groups_)
        std = pd.Series(std_drop).reindex(self.groups_)
        self.scores_["rf_perm"] = score
        self.selected_by_["rf_perm"] = score[(score > 2 * std) & (score > 0)].index.tolist()

    # ------------------------------------------------------ mutual information
    def _mutual_info(self, Xv, yv, rng):
        if len(yv) > self.mi_rows:
            sub = rng.choice(len(yv), self.mi_rows, replace=False)
            Xs, ys = Xv[sub], yv[sub]
        else:
            Xs, ys = Xv, yv
        discrete = np.array([np.unique(Xs[:, j]).size <= 2 for j in range(Xs.shape[1])])
        mi = mutual_info_classif(Xs, ys, discrete_features=discrete, random_state=self.random_state)
        null = mutual_info_classif(Xs, rng.permutation(ys), discrete_features=discrete,
                                   random_state=self.random_state)
        threshold = null.max()                              # best a feature can do by chance
        score = self._by_group(mi).max().reindex(self.groups_)
        self.scores_["mutual_info"] = score
        self.selected_by_["mutual_info"] = score[score > threshold].index.tolist()
        self.mi_null_max_ = float(threshold)

    # ---------------------------------------------- univariate LR likelihood-ratio
    def _univariate(self, Xv, yv):
        n, p1 = len(yv), yv.mean()
        ll0 = n * (p1 * np.log(p1) + (1 - p1) * np.log(1 - p1))
        stats, pvals = {}, {}
        for g, idx in self._gidx.items():
            Xg = StandardScaler().fit_transform(Xv[:, idx])
            rank = int(np.linalg.matrix_rank(Xg - Xg.mean(axis=0)))
            if rank == 0:
                stats[g], pvals[g] = 0.0, 1.0
                continue
            lr = LogisticRegression(C=1e4, max_iter=1000).fit(Xg, yv)
            ll1 = -log_loss(yv, lr.predict_proba(Xg)[:, 1], normalize=False)
            stat = max(2 * (ll1 - ll0), 0.0)
            stats[g], pvals[g] = stat, float(chi2.sf(stat, rank))
        p = pd.Series(pvals).reindex(self.groups_)
        p_adj = pd.Series(benjamini_hochberg(p.values), index=p.index)
        self.scores_["univariate_lrt"] = pd.Series(stats).reindex(self.groups_)
        self.selected_by_["univariate_lrt"] = p_adj[p_adj < self.alpha].index.tolist()

    # ------------------------------------------ random-effects logistic regression
    def _glmm(self, m, Xv, yv, rng, cache):
        """
        Logistic regression with a random intercept per `re_group` level (District), fitted with
        statsmodels' BinomialBayesMixedGLM (MAP / Laplace fit). The grouping variable is NOT a fixed
        effect; every other variable is judged on what it adds given the District effect.
        glmm_wald: joint Wald test per variable, Benjamini-Hochberg adjusted.
        glmm_perm: grouped permutation AUC drop on an inner hold-out (random effects included in the
                   prediction, so the drop is the importance beyond the District effect).
        """
        if "glmm" not in cache:
            try:
                from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
            except ImportError as e:
                raise ImportError("glmm methods need statsmodels (pip install statsmodels)") from e

            re_idx = self._gidx.get(self.re_group)
            if re_idx is None:
                raise ValueError(f"re_group='{self.re_group}' was not found among the encoded columns. "
                                 "It must be in cat_cols and in the data.")
            # cluster of each row = which District dummy is on (rows with none on form their own cluster)
            on = Xv[:, re_idx]
            raw = np.where(on.max(axis=1) > 0, on.argmax(axis=1), len(re_idx))
            levels, codes = np.unique(raw, return_inverse=True)

            # fixed-effect design: every other variable; drop the most frequent dummy of each
            # categorical as the reference level (the intercept would otherwise be redundant)
            cat_set = set(self.cat_cols)
            fixed_cols, fixed_group = [], []
            for g in self.groups_:
                if g == self.re_group:
                    continue
                idx = list(self._gidx[g])
                if g in cat_set and len(idx) > 1:
                    ref = idx[int(np.argmax(Xv[:, idx].mean(axis=0)))]
                    idx = [j for j in idx if j != ref]
                fixed_cols += idx
                fixed_group += [g] * len(idx)
            fixed_cols = np.asarray(fixed_cols, dtype=int)
            fixed_group = np.asarray(fixed_group, dtype=object)

            tr, va = train_test_split(np.arange(len(yv)), test_size=0.25, stratify=yv,
                                      random_state=self.random_state)
            scaler = StandardScaler().fit(Xv[np.ix_(tr, fixed_cols)])
            Xtr = np.column_stack([np.ones(len(tr)), scaler.transform(Xv[np.ix_(tr, fixed_cols)])])
            Ztr = sparse.csr_matrix((np.ones(len(tr)), (np.arange(len(tr)), codes[tr])),
                                    shape=(len(tr), len(levels)))
            ident = np.zeros(len(levels), dtype=int)            # one shared variance for all intercepts
            model = BinomialBayesMixedGLM(yv[tr].astype(float), Xtr, Ztr, ident,
                                          vcp_p=1, fe_p=self.glmm_prior_sd)
            try:
                res, backend = model.fit_map(), "map"           # Laplace: full posterior covariance
            except np.linalg.LinAlgError:
                res, backend = model.fit_vb(), "vb"             # fallback: diagonal covariance only

            k = Xtr.shape[1]
            cov = np.asarray(res.cov_params(), dtype=float)
            if cov.ndim == 1:
                cov = np.diag(cov)
            vc = np.asarray(res.vc_mean, dtype=float)
            sd = float(np.exp(np.asarray(res.vcp_mean, dtype=float)[0]))   # statsmodels stores log(SD)
            self.glmm_backend_ = backend
            self.glmm_n_clusters_ = int(len(levels))
            self.glmm_re_sd_ = sd
            self.glmm_re_sd_empirical_ = float(np.std(vc, ddof=1)) if len(vc) > 1 else float("nan")
            self.glmm_icc_ = sd ** 2 / (sd ** 2 + np.pi ** 2 / 3)          # latent-scale ICC
            self.glmm_converged_ = bool(getattr(res.optim_retvals, "success", True))
            cache["glmm"] = dict(beta=np.asarray(res.fe_mean, dtype=float), cov=cov[:k, :k], vc=vc,
                                 scaler=scaler, fixed_cols=fixed_cols, fixed_group=fixed_group,
                                 codes=codes, va=va)

        c = cache["glmm"]
        beta, fixed_group = c["beta"], c["fixed_group"]
        tested = list(dict.fromkeys(fixed_group))

        if m == "glmm_wald":
            stats, pvals = {}, {}
            for g in tested:
                pos = 1 + np.where(fixed_group == g)[0]               # +1: skip the intercept
                b = beta[pos]
                V = c["cov"][np.ix_(pos, pos)]
                stat = float(max(b @ np.linalg.pinv(V) @ b, 0.0))
                stats[g], pvals[g] = stat, float(chi2.sf(stat, len(pos)))
            p = pd.Series(pvals)
            p_adj = pd.Series(benjamini_hochberg(p.values), index=p.index)
            score = pd.Series(stats).reindex(self.groups_)             # Wald chi-square (District = NaN)
            chosen = p_adj[p_adj < self.alpha].index.tolist()
        else:  # glmm_perm
            va = c["va"]
            Xva = c["scaler"].transform(Xv[np.ix_(va, c["fixed_cols"])])
            yva = yv[va]
            re_va = c["vc"][c["codes"][va]]                            # each stop's District effect
            if len(yva) > self.perm_rows:
                sub = rng.choice(len(yva), self.perm_rows, replace=False)
                Xva, yva, re_va = Xva[sub], yva[sub], re_va[sub]
            b0, b = beta[0], beta[1:]
            base = roc_auc_score(yva, b0 + Xva @ b + re_va)
            mean_drop, std_drop = {}, {}
            for g in tested:
                cols = np.where(fixed_group == g)[0]
                drops = []
                for _ in range(self.perm_repeats):
                    perm = rng.permutation(len(yva))
                    Xq = Xva.copy()
                    Xq[:, cols] = Xva[np.ix_(perm, cols)]              # shuffle the whole variable jointly
                    drops.append(base - roc_auc_score(yva, b0 + Xq @ b + re_va))
                mean_drop[g], std_drop[g] = float(np.mean(drops)), float(np.std(drops))
            score = pd.Series(mean_drop).reindex(self.groups_)
            std = pd.Series(std_drop).reindex(self.groups_)
            chosen = score[(score > 2 * std) & (score > 0)].index.tolist()

        if self.re_keep_group and self.re_group in self.groups_:
            chosen = [self.re_group] + chosen                          # modelled by design, so always kept
        self.scores_[m] = score
        self.selected_by_[m] = chosen

## 4. Build the cross-validation pipeline from the preprocessing pipeline

`build_cv_pipeline` calls the existing builder (`build_simple_pipeline` or `build_pipeline`), keeps all of its **preprocessing steps**, wraps the last encoding step in `NamedTransform`, and inserts the selector before the final model:

`bin → (ordinal → imputer → round →) encode → select → model`

A `joblib.Memory` is passed so that steps shared between pipelines (preprocessing, and the selector when only the final model differs) are fitted once per fold and re-used.

In [ ]:
def build_cv_pipeline(selector, model, kind=PIPELINE_KIND, memory=None):
    if kind == "simple":
        base = build_simple_pipeline(model, cat_cols, bin_cols, scale_cols)
    elif kind == "full":
        base = build_pipeline(model, cat_cols, dis_cols, scale_cols)
    else:
        raise ValueError("kind must be 'simple' or 'full'")

    prep_steps = list(base.steps[:-1])                 # everything except the model
    enc_name, enc = prep_steps[-1]                     # last step = the ColumnTransformer that encodes
    steps = prep_steps[:-1] + [(enc_name, NamedTransform(enc)), ("select", selector), ("model", model)]
    return Pipeline(steps, memory=memory)

## 5. Cross-validation: selection inside every fold

For each selection method and each final model the whole pipeline is passed to `cross_validate`. The same `StratifiedKFold` object is used everywhere, so folds are identical and scores are paired. `return_estimator=True` keeps the fitted pipeline of each fold so we can read which variables each fold selected.

Final models are deliberately generic (L2 logistic regression and gradient boosting) so that the comparison is about the *selection*, not the model.

In [ ]:
CACHE_DIR = tempfile.mkdtemp(prefix="fs_cache_")      # fresh every run, so no stale cache after editing the classes
memory = Memory(CACHE_DIR, verbose=0)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
SCORING = {"roc_auc": "roc_auc", "avg_precision": "average_precision", "neg_brier": "neg_brier_score"}

final_models = {
    "logreg": LogisticRegression(max_iter=2000),
    "hgb": HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200, early_stopping=True,
                                          random_state=SEED),
}

records, fold_auc = [], {}
fold_selected, fold_scores, fold_selectors = {}, {}, {}

for method, cfg in SELECTOR_CONFIGS.items():
    for model_name, model in final_models.items():
        kwargs = dict(cat_cols=tuple(cat_cols), min_votes=CONSENSUS_MIN_VOTES, random_state=SEED)
        kwargs.update(cfg)
        selector = FoldFeatureSelector(**kwargs)
        pipe = build_cv_pipeline(selector, clone(model), memory=memory)
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING,
                             return_estimator=True, n_jobs=1, error_score="raise")

        sels = [est.named_steps["select"] for est in res["estimator"]]
        n_sel = np.array([len(s.selected_groups_) for s in sels])
        fold_auc[(model_name, method)] = res["test_roc_auc"]
        records.append({
            "model": model_name, "selector": method,
            "roc_auc": res["test_roc_auc"].mean(), "roc_auc_sd": res["test_roc_auc"].std(),
            "avg_precision": res["test_avg_precision"].mean(), "avg_precision_sd": res["test_avg_precision"].std(),
            "brier": -res["test_neg_brier"].mean(), "brier_sd": res["test_neg_brier"].std(),
            "n_vars": n_sel.mean(), "n_vars_sd": n_sel.std(),
        })
        if method not in fold_selected:                    # selection does not depend on the final model
            fold_selected[method] = [set(s.selected_groups_) for s in sels]
            fold_scores[method] = [s.scores_.get(s.method) for s in sels]
            fold_selectors[method] = sels
        print(f"{method:15s} {model_name:7s} AUC {res['test_roc_auc'].mean():.4f} ± {res['test_roc_auc'].std():.4f}"
              f" | variables kept: {n_sel.mean():.1f}")

perf = pd.DataFrame(records).set_index(["model", "selector"])

## 6. Results

**Honest performance** (outer folds never influenced the selection) and **paired difference vs. using all variables**.

In [ ]:
show = perf[["roc_auc", "roc_auc_sd", "avg_precision", "avg_precision_sd", "brier", "n_vars", "n_vars_sd"]]
print("Outer-fold performance (mean / SD across folds)")
print(show.to_string())

print("\nPaired ROC-AUC difference vs. 'none' (positive = selection did at least as well as keeping everything)")
rows = []
for (model_name, method), auc in fold_auc.items():
    if method == "none":
        continue
    diff = auc - fold_auc[(model_name, "none")]
    rows.append({"model": model_name, "selector": method, "mean_diff": diff.mean(),
                 "approx_SE": diff.std(ddof=1) / np.sqrt(len(diff)),
                 "n_vars": perf.loc[(model_name, method), "n_vars"]})
print(pd.DataFrame(rows).set_index(["model", "selector"]).to_string())

### Selection stability
For each method: the fraction of outer folds in which each original variable was selected. A variable that is chosen in only 1–2 of 5 folds is not reliably useful, no matter how good one run looks.

In [ ]:
all_groups = sorted({g for sets in fold_selected.values() for s in sets for g in s} |
                    {g for g in X_train.columns})   # includes variables never selected
all_groups = [g for g in all_groups if g in set(feature_group(c, cat_cols) for c in X_train.columns)]

stab = pd.DataFrame({m: {g: np.mean([g in s for s in sets]) for g in all_groups}
                     for m, sets in fold_selected.items() if m != "none"})
stab = stab.loc[stab.mean(axis=1).sort_values(ascending=False).index]
print(stab.to_string())

# average importance score per variable across folds (score scales differ per method, so compare ranks within a column)
mean_scores = {}
for m, lst in fold_scores.items():
    lst = [s for s in lst if s is not None]
    if lst:
        mean_scores[m] = pd.concat(lst, axis=1).mean(axis=1)
mean_scores = pd.DataFrame(mean_scores).reindex(stab.index)
print("\nMean importance score across folds (higher = more important; scales differ by method)")
print(mean_scores.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, max(5, 0.35 * len(stab))))

# (a) stability heatmap
ax = axes[0]
im = ax.imshow(stab.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(stab.shape[1])); ax.set_xticklabels(stab.columns, rotation=45, ha="right")
ax.set_yticks(range(stab.shape[0])); ax.set_yticklabels(stab.index)
ax.set_title("Fraction of outer folds in which a variable was selected")
fig.colorbar(im, ax=ax, fraction=0.04)

# (b) honest AUC per selector, both final models
ax = axes[1]
methods = [m for m in SELECTION_METHODS]
width = 0.38
for i, model_name in enumerate(final_models):
    vals = [perf.loc[(model_name, m), "roc_auc"] for m in methods]
    errs = [perf.loc[(model_name, m), "roc_auc_sd"] for m in methods]
    ax.bar(np.arange(len(methods)) + i * width, vals, width, yerr=errs, capsize=3, label=model_name)
ax.set_xticks(np.arange(len(methods)) + width / 2); ax.set_xticklabels(methods, rotation=45, ha="right")
lo = perf["roc_auc"].min() - 0.02
ax.set_ylim(lo, perf["roc_auc"].max() + 0.01)
ax.set_ylabel("Outer-fold ROC AUC"); ax.set_title("Performance with in-fold selection"); ax.legend()
plt.tight_layout()
plt.savefig("feature_selection_cv.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Does the District random effect matter? (GLMM diagnostics)

The random-effects methods fit a logistic regression with a random intercept per District in every fold.

* **ICC (latent scale)** = share of the variation in the log-odds of a search that sits *between* districts. Rough guide: below about 1% the random effect does almost nothing and ordinary logistic regression is fine; above about 5% districts differ meaningfully and clustering should not be ignored.
* **district_SD** comes from the fitted variance parameter. **district_SD_empirical** is the SD of the fitted district intercepts, as a cross-check; the two should be close. With only 8 districts both are imprecise.
* The table below compares how often each variable is selected with and without the District effect. Large differences point to variables that act mainly as a proxy for location (e.g. `Zip`, `isLA`).

In [ ]:
if "glmm_wald" in fold_selectors:
    diag = pd.DataFrame([{
        "fold": i, "n_districts": s.glmm_n_clusters_, "district_SD": s.glmm_re_sd_,
        "district_SD_empirical": s.glmm_re_sd_empirical_, "ICC_latent": s.glmm_icc_,
        "converged": s.glmm_converged_, "backend": s.glmm_backend_,
    } for i, s in enumerate(fold_selectors["glmm_wald"])])
    print(diag.to_string(index=False))
    print(f"\nMean latent ICC across folds: {diag['ICC_latent'].mean():.2%}")

    cols = [c for c in ["univariate_lrt", "rf_perm", "glmm_wald", "glmm_perm"] if c in stab.columns]
    cmp = stab[cols].copy()
    cmp["glmm_wald - univariate_lrt"] = cmp["glmm_wald"] - cmp["univariate_lrt"]
    cmp["glmm_perm - rf_perm"] = cmp["glmm_perm"] - cmp["rf_perm"]
    print("\nFraction of folds selected (District is kept by construction in the glmm columns):")
    print(cmp.sort_values("glmm_perm", ascending=False).to_string())
else:
    print("No glmm methods in SELECTOR_CONFIGS.")

## 8. Final feature list
The CV above answers *"is this selection procedure any good, and is it stable?"*. The final list for the real model is obtained by running the **same procedure once on the whole training set** (no held-out fold is needed any more). `stable_vars` additionally requires the variable to have been selected in at least `STABLE_FREQ` of the CV folds.

In [ ]:
final_kwargs = dict(cat_cols=tuple(cat_cols), min_votes=CONSENSUS_MIN_VOTES, random_state=SEED)
final_kwargs.update(SELECTOR_CONFIGS[FINAL_METHOD])
final_selector = FoldFeatureSelector(**final_kwargs)
final_pipe = build_cv_pipeline(final_selector, LogisticRegression(max_iter=2000), memory=None)

prep_only = final_pipe[:-1]                 # preprocessing + selector (shares the step objects)
prep_only.fit(X_train, y_train)
fitted_sel = prep_only.named_steps["select"]

full_train_vars = fitted_sel.selected_groups_
stable_vars = stab.index[stab[FINAL_METHOD] >= STABLE_FREQ].tolist() if FINAL_METHOD in stab else []

print(f"Method: {FINAL_METHOD}")
print(f"Selected on the full training set ({len(full_train_vars)}): {full_train_vars}")
print(f"Selected in >= {STABLE_FREQ:.0%} of CV folds ({len(stable_vars)}): {stable_vars}")
print("Only in the full-train selection (unstable):", sorted(set(full_train_vars) - set(stable_vars)))
print("Only stable in CV (not in full-train selection):", sorted(set(stable_vars) - set(full_train_vars)))

out = {
    "method": FINAL_METHOD,
    "pipeline_kind": PIPELINE_KIND,
    "selected_full_train": full_train_vars,
    "stable_vars": stable_vars,
    "stability": stab.round(3).to_dict(),
}
with open("selected_features.json", "w") as f:
    json.dump(out, f, indent=2)
perf.to_csv("feature_selection_cv_performance.csv")
stab.to_csv("feature_selection_stability.csv")
print("\nSaved: selected_features.json, feature_selection_cv_performance.csv, feature_selection_stability.csv")

shutil.rmtree(CACHE_DIR, ignore_errors=True)

## Notes and caveats

* **Rule of thumb for choosing:** prefer the sparsest method whose outer-fold AUC is within about one SE of `none`, and whose variables are stable (selected in most folds).
* `rf_impurity` sums importance over the dummies of a variable, so it is biased toward high-cardinality variables (e.g. `Zip`, `StopDescription`) and tends to keep too much. `rf_perm` (grouped permutation on held-out rows) is the more trustworthy random-forest criterion. `rf_impurity` is kept because it is a common baseline.
* With ~100k+ rows, almost any variable is "statistically significant" in `univariate_lrt`, so it is a weak filter on its own. It is mostly useful as one vote in `consensus`.
* `lasso` at the best-AUC C usually keeps almost everything; `lasso_1se` is the version that actually prunes.
* Selection is done on original variables, not individual dummies, so a categorical is either fully in or fully out.
* The test set is never used. Evaluate on it once, at the end, with a pipeline restricted to the final variables.
* **Random-effects (GLMM) methods:** they mainly change *inference* (which variables matter given the District effect), not accuracy: the final models already see District as a feature, and with thousands of stops per district partial pooling barely differs from fixed dummies. A random intercept captures baseline shifts between districts only, not spatial dependence between neighbouring districts or within a district. With 8 districts the variance of the random effect is imprecise. `glmm_wald` is a lenient filter at this sample size; `glmm_perm` is the stricter one. District is kept by construction in both.
* If the model must work in districts it has not seen, evaluate with grouped CV (e.g. `StratifiedGroupKFold` with District as the group); ordinary stratified folds put neighbouring stops in both train and validation.
* Other techniques worth trying if you need more: Boruta / shadow-feature selection, RFECV with logistic regression, stability selection (lasso on many bootstrap subsamples), SHAP importance from a boosted model.